# 01 · Dati e feature / Data and features

**Input · IT:** `data/market_data.csv`: 3.868 righe × 19 colonne, SPY (azioni USA), TLT (Treasury USA a lunga scadenza), GLD (oro).

**Output · IT:** Prezzi allineati, controlli sui dati e `results/features.csv` con 15 feature candidate.

**Input · EN:** `data/market_data.csv`: 3,868 rows × 19 columns, SPY (US equities), TLT (long-term US Treasuries), GLD (gold).

**Output · EN:** Aligned prices, data checks and `results/features.csv` with 15 candidate features.

**Prima di iniziare / Before starting:** [pandas: date e selezione / dates and indexing](https://pandas.pydata.org/docs/user_guide/indexing.html), [finestre mobili / rolling windows](https://pandas.pydata.org/docs/user_guide/window.html); rendimenti semplici e prezzi adjusted / simple returns and adjusted prices. Le finestre contano osservazioni / Windows count observations.

**Avvio / Start:** Jupyter dalla cartella del progetto; import di `core` dalla radice con `settings.json` / Jupyter from the project folder; import `core` from the root containing `settings.json`.


## A · Preparazione dei dati / Data preparation

### 1. Caricare il CSV / Load the CSV

**IT:** Implementate `data.load_data` leggendo date, numeri e tutte le 19 colonne dal CSV unico. Controllate schema, duplicati, valori mancanti e prezzi non positivi senza modificare la sorgente.

**EN:** Implement `data.load_data` to read dates, numeric values and all 19 columns from the single CSV. Check the schema, duplicates, missing values and nonpositive prices without changing the source.


In [17]:
from pathlib import Path
import sys
import pandas as pd
import json

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from core.data import load_data

settings = json.loads((project_root / "settings.json").read_text())
source_path = project_root / settings["data_file"]
raw = load_data(source_path)
display(pd.Series({
    "righe": len(raw), "colonne": raw.shape[1],
    "prima data": raw["date"].min(), "ultima data": raw["date"].max(),
    "date duplicate": raw["date"].duplicated().sum(),
    "valori mancanti": raw.isna().sum().sum(),
}))
display(raw.head())

righe                             3868
colonne                             19
prima data         2004-11-18 00:00:00
ultima data        2020-04-01 00:00:00
date duplicate                       0
valori mancanti                      0
dtype: object

,date,spy_open,spy_high,spy_low,spy_close,spy_adj_close,spy_volume,tlt_open,tlt_high,tlt_low,tlt_close,tlt_adj_close,tlt_volume,gld_open,gld_high,gld_low,gld_close,gld_adj_close,gld_volume
0,2004-11-18,118.529999,118.800003,118.230003,118.739998,86.974640,31854300,88.550003,89.059998,88.410004,88.970001,52.950603,891500,44.430000,44.490002,44.070000,44.380001,44.380001,5992000
1,2004-11-19,118.699997,118.720001,117.139999,117.419998,86.007759,54276500,88.900002,89.029999,88.089996,88.260002,52.528084,1244500,44.490002,44.919998,44.470001,44.779999,44.779999,11655300
2,2004-11-22,117.169998,118.120003,117.029999,117.980003,86.417976,37560200,88.349998,88.870003,88.300003,88.720001,52.801838,511500,44.750000,44.970001,44.740002,44.950001,44.950001,11996000
3,2004-11-23,117.930000,118.260002,117.370003,118.160004,86.549820,41968800,88.500000,89.199997,88.480003,88.830002,52.867249,746300,44.880001,44.919998,44.720001,44.750000,44.750000,3169200
4,2004-11-24,118.269997,118.589996,118.050003,118.440002,86.754906,29724800,88.910004,88.910004,88.730003,88.830002,52.867249,309700,44.930000,45.049999,44.790001,45.049999,45.049999,6105100


### 2. Preparare i prezzi / Prepare prices

**IT:** Implementate `data.prepare_prices` convertendo `spy_adj_close`, `tlt_adj_close`, `gld_adj_close` in `equity`, `bond`, `gold`. Restituite tre colonne con indice `date` crescente e univoco, senza riempire valori mancanti.

**EN:** Implement `data.prepare_prices` by mapping `spy_adj_close`, `tlt_adj_close`, `gld_adj_close` to `equity`, `bond`, `gold`. Return three columns with an increasing, unique `date` index and no filled missing values.


In [ ]:
from core.data import prepare_prices

prepare_prices(df)

### 3. Esplorare il campione / Explore the sample

**IT:** Calcolate rendimenti giornalieri, statistiche e distribuzioni; tracciate i prezzi ribasati a 100. Verificate unità, date e primo rendimento mancante, usando gli altri campi OHLCV solo quando servono.

**EN:** Calculate daily returns, summary statistics and distributions; plot prices rebased to 100. Check units, dates and the missing first return, using the other OHLCV fields when useful.


### 4. Controllare anomalie e relazioni / Check anomalies and relationships

**IT:** Segnalate le variazioni estreme e calcolate le correlazioni dei rendimenti allineati. Restituite righe da controllare e una matrice leggibile, conservando i movimenti di mercato autentici.

**EN:** Flag extreme changes and calculate correlations between aligned returns. Return rows for review and a readable matrix, retaining genuine market moves.


### 5. Integrare i controlli / Integrate the checks

**IT:** Portate lettura e preparazione in `core/data.py`, i grafici riutilizzabili in `core/plots.py`. Richiamate le funzioni dal notebook e verificate che input validi passino e duplicati o prezzi invalidi siano respinti.

**EN:** Move loading and preparation into `core/data.py`, and reusable charts into `core/plots.py`. Call the functions from the notebook and check that valid input passes while duplicate dates or invalid prices are rejected.


OK - date duplicate: Date mancanti o duplicate.
OK - prezzo 0.0: I prezzi devono essere strettamente positivi.
OK - prezzo -1.0: I prezzi devono essere strettamente positivi.
OK - prezzo nan: Valori numerici mancanti o non finiti.
OK - prezzo inf: Valori numerici mancanti o non finiti.


Tre grafici EDA esportati in results/figures


## B · Le 15 feature candidate / The 15 candidate features

### 6. Rendimenti e volatilità / Returns and volatility

**IT:** Calcolate rendimento semplice e volatilità per ciascun asset: sei colonne, con volatilità su 20 rendimenti e annualizzazione dai settings. Controllate deviazione standard campionaria, warmup e volatilità non negativa.

**EN:** Calculate simple return and volatility for each asset: six columns, using 20 returns and the annualization in settings. Check the sample standard deviation, warmup and nonnegative volatility.


### 7. Momentum e drawdown / Momentum and drawdown

**IT:** Calcolate momentum su 60 intervalli e drawdown dal massimo di 252 prezzi per ciascun asset: altre sei colonne. Usate le finestre dei settings e verificate estremi delle finestre e drawdown mai positivo.

**EN:** Calculate 60-interval momentum and drawdown from the maximum of 252 prices for each asset: six more columns. Use the windows in settings and check their endpoints and nonpositive drawdowns.


### 8. Correlazioni mobili / Rolling correlations

**IT:** Calcolate le tre correlazioni Pearson su 60 rendimenti: equity–bond, equity–gold e bond–gold. Controllate date, intervallo da −1 a 1 e finestre costanti senza assegnare valori sostitutivi.

**EN:** Calculate three Pearson correlations over 60 returns: equity–bond, equity–gold and bond–gold. Check dates, the −1 to 1 range and constant windows without inserting replacement values.


### 9. Assemblare le feature / Assemble the features

**IT:** Implementate `features.build_features`, applicate il warmup comune ed esportate `results/features.csv`. Controllate 15 colonne, nomi con prefisso dell’asset, date condivise e valori finiti dopo il warmup.

**EN:** Implement `features.build_features`, apply the common warmup and export `results/features.csv`. Check 15 columns, asset-prefixed names, shared dates and finite values after warmup.


### 10. Verificare il timing / Check timing

**IT:** Portate le feature in `core/features.py` e ricostruitele da un prefisso di prezzi. Verificate che aggiungere o cambiare prezzi futuri lasci identiche tutte le feature precedenti.

**EN:** Move feature construction into `core/features.py` and rebuild from a price prefix. Check that adding or changing future prices leaves all earlier features identical.


## Note di lettura e passaggio al notebook 02

- Rendimenti e momentum sono decimali; la volatilità è annualizzata con 252 sedute e deviazione standard campionaria (`ddof=1`). Il drawdown è rispetto al massimo mobile di 252 prezzi, non al massimo storico dell'intero campione.
- Ogni feature alla data *t* usa il prezzo di chiusura adjusted di *t* e dati precedenti. È disponibile dopo la chiusura di *t*: per un'azione precedente alla chiusura occorre ritardare gli input.
- Le prime 251 righe servono al warm-up comune. Non vengono eliminate silenziosamente eventuali righe non finite successive: la costruzione segnala un errore, ad esempio per correlazioni non definite su finestre costanti.
- I grafici e le statistiche sull'intero campione sono descrittivi. Nel notebook 02 split cronologico 60/20/20 dopo il warm-up, selezione e scaler stimati soltanto sul training. Questa fase non sceglie feature o modelli in base all'EDA dell'intero campione.
- TLT rappresenta Treasury a lunga scadenza. Il dataset finisce il 1 aprile 2020; le statistiche non descrivono la successiva ripresa o il 2022.
